In [1]:
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from torchvision import datasets, transforms
import pandas as pd
import numpy as np

In [2]:
data_path = Path("data")
df = pd.read_csv(data_path/"fashion-mnist_test.csv", dtype= np.float32)

In [3]:
y = df.iloc[:,0].values

In [4]:
X = df.drop("label", axis= 1).values

In [5]:
y

array([0., 1., 2., ..., 8., 8., 1.], shape=(10000,), dtype=float32)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42, shuffle= True)

In [7]:
X_train = torch.tensor(X_train, dtype=torch.float32) / 255.0
X_test = torch.tensor(X_test, dtype=torch.float32) / 255.0

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [8]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

torch.Size([8000, 784]) torch.Size([2000, 784]) torch.Size([8000]) torch.Size([2000])


In [9]:
X_train = X_train.view(-1,1,28,28)
X_test = X_test.view(-1,1,28,28)

In [10]:
data_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p= 0.4),
    transforms.AutoAugment(),
    transforms.ToTensor()
])

In [11]:
train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

In [12]:
import os
NUM_WORKERS = os.cpu_count()
BATCH_SIZE = 32

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [13]:
class FashionCNN(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.layer_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=1, 
                  out_channels=hidden_units,
                  kernel_size=(3,3),
                  stride=1,
                  padding=1
                 ),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size= (2,2),
            stride= 2
        ) )

        self.layer_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_units, 
                  out_channels=hidden_units,
                  kernel_size=(3,3),
                  stride=1,
                  padding=1
                 ),
        nn.ReLU(),
        nn.MaxPool2d(
            kernel_size= (2,2),
            stride= 2
        ))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=6272, 
                      out_features=10)
        )

        
            
    def forward(self, x):
        return self.classifier(self.layer_block_2(self.layer_block_1(x)))
              
                  
                  

In [14]:
torch.manual_seed(42)
model = FashionCNN(
    input_shape=3,
    hidden_units=128,
    output_shape=10
)

In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params= model.parameters(), lr=0.001)

In [16]:
def train_step(model: torch.nn.Module, 
               dataloader: torch.utils.data.DataLoader, 
               loss_fn: torch.nn.Module, 
               optimizer: torch.optim.Optimizer):
    # Put model in train mode
    model.train()
    
    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0
    
    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item() 

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metrics across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch 
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [17]:
def test_step(model: torch.nn.Module, 
              dataloader: torch.utils.data.DataLoader, 
              loss_fn: torch.nn.Module):
    # Put model in eval mode
    model.eval() 
    
    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0
    
    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
    
            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
            
    # Adjust metrics to get average loss and accuracy per batch 
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [18]:
def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5):
    
    # 2. Create empty results dictionary
    results = {"train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    
    # 3. Loop through training and testing steps for a number of epochs
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn)
        
        # 4. Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc * 100:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc * 100:.4f}"
        )

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    # 6. Return the filled results at the end of the epochs
    return results

In [19]:
torch.manual_seed(42) 
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 10

# Recreate an instance of model
model = FashionCNN(input_shape=1,
                  hidden_units=128, 
                  output_shape=10
                  )

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

# Train model_0 
model_results = train(model=model, 
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn, 
                        epochs=NUM_EPOCHS)

Epoch: 1 | train_loss: 0.6713 | train_acc: 75.9250 | test_loss: 0.4508 | test_acc: 83.0853
Epoch: 2 | train_loss: 0.4009 | train_acc: 85.6625 | test_loss: 0.3916 | test_acc: 86.5575
Epoch: 3 | train_loss: 0.3376 | train_acc: 87.6875 | test_loss: 0.3821 | test_acc: 86.2599
Epoch: 4 | train_loss: 0.2919 | train_acc: 89.4500 | test_loss: 0.3664 | test_acc: 86.7560
Epoch: 5 | train_loss: 0.2494 | train_acc: 90.9750 | test_loss: 0.4011 | test_acc: 87.5496
Epoch: 6 | train_loss: 0.2246 | train_acc: 91.9000 | test_loss: 0.3592 | test_acc: 88.3433
Epoch: 7 | train_loss: 0.1976 | train_acc: 93.1250 | test_loss: 0.3552 | test_acc: 87.6984
Epoch: 8 | train_loss: 0.1792 | train_acc: 93.6250 | test_loss: 0.3212 | test_acc: 89.3353
Epoch: 9 | train_loss: 0.1508 | train_acc: 94.6125 | test_loss: 0.3697 | test_acc: 88.0952
Epoch: 10 | train_loss: 0.1444 | train_acc: 94.8750 | test_loss: 0.3675 | test_acc: 89.0377


In [20]:
# 1. Klasör yoksa oluşturuyoruz
model_folder = "model"
if not os.path.exists(model_folder):
    os.makedirs(model_folder)
    print(f"'{model_folder}' klasörü oluşturuldu.")

# 2. Kayıt yolunu (path) belirliyoruz
MODEL_SAVE_PATH = os.path.join(model_folder, "fashion_mnist_model.pth")

# 3. Kaydetme işlemini gerçekleştiriyoruz
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH)

print(f"Model başarıyla şuraya kaydedildi: {MODEL_SAVE_PATH}")

Model başarıyla şuraya kaydedildi: model\fashion_mnist_model.pth
